# 02 — Treinamento dos seis experimentos

> **Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.

| Exp. | Modelo | Resolução | Pesos de classe |
|---|---|---|---|
| E1 | MLP | 128 × 96 | Não |
| E2 | MLP | 128 × 96 | Sim |
| E3 | CNN | 128 × 96 | Não |
| E4 | CNN | 128 × 96 | Sim |
| E5 | CNN | 256 × 192 | Não |
| E6 | CNN | 512 × 384 | Não |

Cada experimento salva em `outputs/<id>/`: `model.keras`, `history.csv`, `summary.json` (configuração, custo e métricas de
**validação**). O **teste não é usado aqui** — ele só é avaliado no notebook 03, depois de todas as escolhas.

Se o Colab desconectar, basta rodar de novo: experimentos já concluídos são pulados (use `force=True` para refazer).

In [4]:
# Configuração do ambiente
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Caminho da pasta trashnet-rna no Google Drive. Se ele não existir na sua conta,
    # o projeto é procurado automaticamente no Drive (pela pasta src/trashnet).
    PROJECT_DIR = "/content/drive/MyDrive/Faculdade/UNIFOR/IA/Redes Neurais/trashnet-rna"
    if not os.path.isdir(os.path.join(PROJECT_DIR, "src", "trashnet")):
        achados = sorted(Path("/content/drive/MyDrive").glob("**/src/trashnet/config.py"))
        if achados:
            PROJECT_DIR = str(achados[0].parents[2])
    # As imagens ficam no disco do próprio Colab, que é mais rápido que o Drive.
    os.environ["TRASHNET_DATA_DIR"] = "/content/data"
else:
    PROJECT_DIR = ".."  # rodando no computador, a partir da pasta notebooks/
if not os.path.isdir(os.path.join(PROJECT_DIR, "src", "trashnet")):
    raise FileNotFoundError(f"Não encontrei o projeto em {PROJECT_DIR}. Corrija o PROJECT_DIR acima.")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
print("Projeto:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Projeto: /content/drive/MyDrive/Faculdade/UNIFOR/IA/Redes Neurais/trashnet-rna


In [16]:
!nvidia-smi -L || echo "Sem GPU — o treino vai funcionar, mas bem mais devagar."

GPU 0: Tesla T4 (UUID: GPU-d30598e4-95dd-7a43-3f72-3b1590b68ecf)


In [6]:
import json, pandas as pd
from trashnet import config as C
from trashnet.data import download_dataset, load_split, split_summary
from trashnet.train import run_experiment, summary_table, hardware_info

download_dataset()
split_df = load_split()          # mesma divisão para todos os experimentos
display(split_summary(split_df))
print(json.dumps(hardware_info(), indent=2, ensure_ascii=False))

Baixando https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip?download=true
Dataset extraído em /content/data/dataset-resized


split,train,val,test,total
label,,,,
glass,348,75,75,498
paper,416,89,89,594
cardboard,282,60,61,403
plastic,336,72,72,480
metal,286,61,62,409
trash,96,21,20,137
total,1764,378,379,2521


{
  "gpu": "Tesla T4",
  "cpu": "x86_64",
  "ram_total_gb": 12.7,
  "tensorflow": "2.21.0",
  "keras": "3.13.2"
}


In [7]:
print("MLP:", C.MLP_PARAMS)
print("CNN:", C.CNN_PARAMS)
print("Treino:", C.TRAIN_PARAMS)
pd.DataFrame(C.EXPERIMENTS).T

MLP: {'hidden_units': [256, 128], 'dropout': 0.3, 'learning_rate': 0.0001}
CNN: {'filters': [32, 64, 128, 256], 'kernel_size': 3, 'dense_units': 64, 'dropout': 0.3, 'learning_rate': 0.001}
Treino: {'optimizer': 'adam', 'loss': 'sparse_categorical_crossentropy', 'batch_size': 32, 'max_epochs': 100, 'early_stopping_monitor': 'val_loss', 'early_stopping_patience': 10}


,model,size,class_weights
E1_mlp_128x96,mlp,"(128, 96)",False
E2_mlp_128x96_pesos,mlp,"(128, 96)",True
E3_cnn_128x96,cnn,"(128, 96)",False
E4_cnn_128x96_pesos,cnn,"(128, 96)",True
E5_cnn_256x192,cnn,"(256, 192)",False
E6_cnn_512x384,cnn,"(512, 384)",False


## Experimentos (rodem na ordem; cada célula é independente)

In [8]:
resumos = {}

### E1_mlp_128x96 — MLP, 128×96, sem pesos de classe

In [9]:
resumos["E1_mlp_128x96"] = run_experiment("E1_mlp_128x96")

Model: "mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,471,110 (36.13 MB)

 Trainable params: 9,471,110 (36.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
56/56 - 7s - 130ms/step - accuracy: 0.2063 - loss: 2.1988 - val_accuracy: 0.2646 - val_loss: 1.7213
Epoch 2/100
56/56 - 0s - 6ms/step - accuracy: 0.2279 - loss: 1.7495 - val_accuracy: 0.2672 - val_loss: 1.7276
Epoch 3/100
56/56 - 1s - 14ms/step - accuracy: 0.2347 - loss: 1.7326 - val_accuracy: 0.2857 - val_loss: 1.7092
Epoch 4/100
56/56 - 1s - 15ms/step - accuracy: 0.2472 - loss: 1.7267 - val_accuracy: 0.2884 - val_loss: 1.6808
Epoch 5/100
56/56 - 1s - 15ms/step - accuracy: 0.2449 - loss: 1.6998 - val_accuracy: 0.2672 - val_loss: 1.6770
Epoch 6/100
56/56 - 1s - 15ms/step - accuracy: 0.2551 - loss: 1.6694 - val_accuracy: 0.2831 - val_loss: 1.6126
Epoch 7/100
56/56 - 1s - 15ms/step - accuracy: 0.2398 - loss: 1.6712 - val_accuracy: 0.3016 - val_loss: 1.6091
Epoch 8/100
56/56 - 1s - 15ms/step - accuracy: 0.2602 - loss: 1.6540 - val_accuracy: 0.3228 - val_loss: 1.6059
Epoch 9/100
56/56 - 1s - 15ms/step - accuracy: 0.2846 - loss: 1.6408 - val_accuracy: 0.2963 - val_loss: 1.5697
E

### E2_mlp_128x96_pesos — MLP, 128×96, **com** pesos de classe

In [10]:
resumos["E2_mlp_128x96_pesos"] = run_experiment("E2_mlp_128x96_pesos")

Model: "mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,471,110 (36.13 MB)

 Trainable params: 9,471,110 (36.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
56/56 - 5s - 93ms/step - accuracy: 0.1842 - loss: 2.2905 - val_accuracy: 0.1958 - val_loss: 1.7649
Epoch 2/100
56/56 - 0s - 7ms/step - accuracy: 0.1570 - loss: 1.7901 - val_accuracy: 0.1561 - val_loss: 1.7866
Epoch 3/100
56/56 - 0s - 7ms/step - accuracy: 0.1468 - loss: 1.7843 - val_accuracy: 0.2354 - val_loss: 1.7929
Epoch 4/100
56/56 - 0s - 6ms/step - accuracy: 0.1831 - loss: 1.7827 - val_accuracy: 0.2328 - val_loss: 1.7807
Epoch 5/100
56/56 - 0s - 6ms/step - accuracy: 0.1859 - loss: 1.7729 - val_accuracy: 0.2434 - val_loss: 1.7747
Epoch 6/100
56/56 - 1s - 14ms/step - accuracy: 0.2154 - loss: 1.7567 - val_accuracy: 0.2037 - val_loss: 1.7462
Epoch 7/100
56/56 - 0s - 6ms/step - accuracy: 0.2239 - loss: 1.7389 - val_accuracy: 0.2407 - val_loss: 1.7523
Epoch 8/100
56/56 - 1s - 15ms/step - accuracy: 0.2336 - loss: 1.7415 - val_accuracy: 0.2857 - val_loss: 1.7456
Epoch 9/100
56/56 - 1s - 14ms/step - accuracy: 0.2296 - loss: 1.7168 - val_accuracy: 0.1693 - val_loss: 1.7493
Epoch 

### E3_cnn_128x96 — CNN, 128×96, sem pesos de classe

In [11]:
resumos["E3_cnn_128x96"] = run_experiment("E3_cnn_128x96")

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 96, 128, 32)    │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 48, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 24, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,254 (1.55 MB)

 Trainable params: 405,254 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
56/56 - 18s - 329ms/step - accuracy: 0.2211 - loss: 1.7275 - val_accuracy: 0.3148 - val_loss: 1.6653
Epoch 2/100
56/56 - 1s - 21ms/step - accuracy: 0.2897 - loss: 1.6398 - val_accuracy: 0.3360 - val_loss: 1.6056
Epoch 3/100
56/56 - 1s - 19ms/step - accuracy: 0.3866 - loss: 1.4941 - val_accuracy: 0.4550 - val_loss: 1.3531
Epoch 4/100
56/56 - 1s - 19ms/step - accuracy: 0.4337 - loss: 1.4176 - val_accuracy: 0.4735 - val_loss: 1.3378
Epoch 5/100
56/56 - 1s - 18ms/step - accuracy: 0.4620 - loss: 1.3387 - val_accuracy: 0.4577 - val_loss: 1.3445
Epoch 6/100
56/56 - 1s - 19ms/step - accuracy: 0.4671 - loss: 1.3153 - val_accuracy: 0.4788 - val_loss: 1.2332
Epoch 7/100
56/56 - 1s - 17ms/step - accuracy: 0.4892 - loss: 1.2923 - val_accuracy: 0.4762 - val_loss: 1.2760
Epoch 8/100
56/56 - 1s - 19ms/step - accuracy: 0.4830 - loss: 1.2817 - val_accuracy: 0.4947 - val_loss: 1.2060
Epoch 9/100
56/56 - 1s - 17ms/step - accuracy: 0.5102 - loss: 1.2490 - val_accuracy: 0.5106 - val_loss: 1.2318

### E4_cnn_128x96_pesos — CNN, 128×96, **com** pesos de classe

In [12]:
resumos["E4_cnn_128x96_pesos"] = run_experiment("E4_cnn_128x96_pesos")

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 128, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 96, 128, 32)    │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 48, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 24, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,254 (1.55 MB)

 Trainable params: 405,254 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
56/56 - 9s - 155ms/step - accuracy: 0.1803 - loss: 1.7886 - val_accuracy: 0.2751 - val_loss: 1.7618
Epoch 2/100
56/56 - 1s - 20ms/step - accuracy: 0.2392 - loss: 1.7552 - val_accuracy: 0.3122 - val_loss: 1.6025
Epoch 3/100
56/56 - 1s - 19ms/step - accuracy: 0.3260 - loss: 1.6352 - val_accuracy: 0.4153 - val_loss: 1.5055
Epoch 4/100
56/56 - 1s - 18ms/step - accuracy: 0.2954 - loss: 1.6298 - val_accuracy: 0.3466 - val_loss: 1.5639
Epoch 5/100
56/56 - 1s - 19ms/step - accuracy: 0.3146 - loss: 1.6010 - val_accuracy: 0.4259 - val_loss: 1.4983
Epoch 6/100
56/56 - 1s - 19ms/step - accuracy: 0.3702 - loss: 1.5308 - val_accuracy: 0.4127 - val_loss: 1.4360
Epoch 7/100
56/56 - 1s - 18ms/step - accuracy: 0.3498 - loss: 1.5603 - val_accuracy: 0.4312 - val_loss: 1.4536
Epoch 8/100
56/56 - 1s - 19ms/step - accuracy: 0.3736 - loss: 1.4738 - val_accuracy: 0.4365 - val_loss: 1.3820
Epoch 9/100
56/56 - 1s - 18ms/step - accuracy: 0.3980 - loss: 1.4511 - val_accuracy: 0.4259 - val_loss: 1.3867


### E5_cnn_256x192 — CNN, 256×192, sem pesos de classe

In [13]:
resumos["E5_cnn_256x192"] = run_experiment("E5_cnn_256x192")

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 192, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 192, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 192, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 96, 128, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 96, 128, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 48, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 48, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 24, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,254 (1.55 MB)

 Trainable params: 405,254 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
56/56 - 23s - 409ms/step - accuracy: 0.2262 - loss: 1.7256 - val_accuracy: 0.3307 - val_loss: 1.7005
Epoch 2/100
56/56 - 3s - 62ms/step - accuracy: 0.3112 - loss: 1.6071 - val_accuracy: 0.3836 - val_loss: 1.4765
Epoch 3/100
56/56 - 3s - 62ms/step - accuracy: 0.3855 - loss: 1.5069 - val_accuracy: 0.4603 - val_loss: 1.3366
Epoch 4/100
56/56 - 3s - 62ms/step - accuracy: 0.4280 - loss: 1.4144 - val_accuracy: 0.4735 - val_loss: 1.3081
Epoch 5/100
56/56 - 3s - 62ms/step - accuracy: 0.4518 - loss: 1.3839 - val_accuracy: 0.4762 - val_loss: 1.2866
Epoch 6/100
56/56 - 3s - 60ms/step - accuracy: 0.4717 - loss: 1.3294 - val_accuracy: 0.4312 - val_loss: 1.4007
Epoch 7/100
56/56 - 3s - 62ms/step - accuracy: 0.4785 - loss: 1.3344 - val_accuracy: 0.5000 - val_loss: 1.2798
Epoch 8/100
56/56 - 3s - 62ms/step - accuracy: 0.4858 - loss: 1.2915 - val_accuracy: 0.5079 - val_loss: 1.2074
Epoch 9/100
56/56 - 3s - 60ms/step - accuracy: 0.4994 - loss: 1.2729 - val_accuracy: 0.5159 - val_loss: 1.2350

### E6_cnn_512x384 — CNN, 512×384, sem pesos de classe (o mais demorado)

In [14]:
resumos["E6_cnn_512x384"] = run_experiment("E6_cnn_512x384")

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 384, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 384, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 384, 512, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 192, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 192, 256, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 96, 128, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 96, 128, 128)   │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 48, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 48, 64, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 24, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,254 (1.55 MB)

 Trainable params: 405,254 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
56/56 - 46s - 830ms/step - accuracy: 0.2336 - loss: 1.7165 - val_accuracy: 0.3042 - val_loss: 1.6110
Epoch 2/100
56/56 - 14s - 256ms/step - accuracy: 0.3254 - loss: 1.5802 - val_accuracy: 0.4286 - val_loss: 1.4594
Epoch 3/100
56/56 - 14s - 255ms/step - accuracy: 0.4036 - loss: 1.4683 - val_accuracy: 0.4286 - val_loss: 1.3933
Epoch 4/100
56/56 - 14s - 255ms/step - accuracy: 0.4161 - loss: 1.4382 - val_accuracy: 0.4339 - val_loss: 1.3927
Epoch 5/100
56/56 - 14s - 255ms/step - accuracy: 0.4456 - loss: 1.4128 - val_accuracy: 0.4762 - val_loss: 1.3088
Epoch 6/100
56/56 - 14s - 255ms/step - accuracy: 0.4694 - loss: 1.3600 - val_accuracy: 0.4735 - val_loss: 1.2984
Epoch 7/100
56/56 - 13s - 236ms/step - accuracy: 0.4717 - loss: 1.3459 - val_accuracy: 0.4497 - val_loss: 1.3032
Epoch 8/100
56/56 - 14s - 257ms/step - accuracy: 0.4966 - loss: 1.2975 - val_accuracy: 0.4974 - val_loss: 1.2336
Epoch 9/100
56/56 - 14s - 255ms/step - accuracy: 0.5153 - loss: 1.2709 - val_accuracy: 0.5185 - 

## Resumo (lido do disco)

In [15]:
resumo = summary_table()   # não treina de novo: apenas lê outputs/*/summary.json
C.RESULTS_DIR.mkdir(exist_ok=True)
resumo.to_csv(C.RESULTS_DIR / "resumo_treino_validacao.csv", index=False)
resumo

,exp_id,rotulo,parametros,epocas_executadas,melhor_epoca,tempo_total_s,tempo_medio_epoca_s,pico_ram_mb,pico_gpu_mb,val_accuracy,val_f1_macro
0,E1_mlp_128x96,MLP 128×96 sem pesos,9471110,34,24,25.3,0.73,1783,329,0.3439,0.2400
1,E2_mlp_128x96_pesos,MLP 128×96 com pesos,9471110,39,29,24.8,0.63,1849,411,0.2884,0.1602
2,E3_cnn_128x96,CNN 128×96 sem pesos,405254,57,47,75.8,1.32,2180,1012,0.7249,0.7122
3,E4_cnn_128x96_pesos,CNN 128×96 com pesos,405254,61,51,71.6,1.17,2237,617,0.7275,0.7097
4,E5_cnn_256x192,CNN 256×192 sem pesos,405254,62,52,237.5,3.82,3119,2842,0.7302,0.6861
5,E6_cnn_512x384,CNN 512×384 sem pesos,405254,93,83,1299.1,13.94,5127,7969,0.7698,0.7271
